# 09 — ColBERT-Extended Eval Harness: Generation

Extends the strategy-comparison generation loop to run every working strategy against **both** retrieval backends — the dense Chroma store (`chroma_naive_gemini`) and the PyLate ColBERT index built in `08_colbert.ipynb` — instead of dense only.

**Keys change from the dense-only generation notebook:** rows are keyed by `(strategy, backend, id)` instead of `(strategy, id)`, and everything writes to a new checkpoint file, `eval_results_backend.jsonl`, so the original `eval_results.jsonl` is left untouched.

**Before running this:** this doubles the number of (strategy, backend) combinations from 6 to 12, which doubles LLM generation calls against whatever quota you point `llm` at. Given how hard the 200k TPD Groq ceiling was hit during the dense-only run, strongly consider running this against the local Ollama model (`qwen2.5:7b`) instead of Groq — see the LLM selection cell below. The retrieval side (both backends) costs no LLM tokens at all; only generation does.

**One assumption flagged up front:** this notebook doesn't have your actual `rag_lab/strategies/*.py` source in front of it, so it can't see whether each strategy class exposes a way to retrieve the *exact* context it used internally (multi_query, rag_fusion, step_back and decomposition all retrieve via more than one sub-query and fuse/dedupe before generating — the contexts RAGAS needs for `context_precision` / `context_recall` are whatever list each strategy actually fed its prompt, not necessarily a plain single-query retriever call). The capture helper below tries, in order: `strategy.last_contexts` (if your classes cache it after `.run()`), then `strategy.retriever.invoke(query)` (if they expose their fused retriever), then falls back to a plain `vectorstore.as_retriever().invoke(query)` call with a logged warning. Each row records which tier was used (`context_source`), so you can spot-check before trusting `context_precision`/`context_recall` on the fallback-tier rows.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
import getpass
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass.getpass("Enter Groq API Key: ")
if not os.environ.get('GOOGLE_API_KEY'):
    os.environ['GOOGLE_API_KEY'] = getpass.getpass("Enter Google API Key: ")

Enter Groq API Key:  ········
Enter Google API Key:  ········


## Load both backends against the clean, deduplicated 576-chunk corpus

In [2]:
print(project_root)

/home/mohnish/Codes/rag-lab


In [3]:
import json
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print(f"project_root resolved to: {project_root}")  # sanity-check this before trusting the paths below

SOURCE = project_root / "eval" / "eval_results.jsonl"
DEST = project_root / "eval" / "eval_results_backend.jsonl"

STRATEGY_TO_BACKEND = {
    "naive": "dense",
    "colbert": "colbert",
}

# What's already in the destination, so this is idempotent.
existing = set()
if DEST.exists():
    with open(DEST) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            existing.add((row.get("strategy"), row.get("backend"), row.get("id")))

backfilled = 0
skipped_already_present = 0
skipped_no_answer = 0

with open(SOURCE) as f:
    source_rows = [json.loads(line) for line in f if line.strip()]

with open(DEST, "a") as out:
    for row in source_rows:
        src_strategy = row.get("strategy")
        if src_strategy not in STRATEGY_TO_BACKEND:
            continue  # multi_query/rag_fusion/hyde/step_back/decomposition stay untouched -- not redundant
        if row.get("answer") is None:
            skipped_no_answer += 1
            continue

        backend = STRATEGY_TO_BACKEND[src_strategy]
        key = ("naive", backend, row["id"])
        if key in existing:
            skipped_already_present += 1
            continue

        out.write(json.dumps({
            "strategy": "naive",
            "backend": backend,
            "id": row["id"],
            "question": row["question"],
            "question_type": row.get("question_type"),
            "ground_truth": row.get("ground_truth"),
            "answer": row["answer"],
            "contexts": row.get("contexts"),
            "error": None,
        }) + "\n")
        existing.add(key)
        backfilled += 1

print(f"Backfilled {backfilled} rows into {DEST.name}")
print(f"Skipped {skipped_already_present} already present, {skipped_no_answer} with no answer")


project_root resolved to: /home/mohnish/Codes/rag-lab
Backfilled 0 rows into eval_results_backend.jsonl
Skipped 125 already present, 1 with no answer


In [4]:
import json
import time
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

dense_vectorstore = Chroma(
    embedding_function=embeddings,
    persist_directory="~/Codes/rag-lab/data/chroma_naive_gemini",
)
print(f"dense backend: {dense_vectorstore._collection.count()} chunks")

dense backend: 576 chunks


In [5]:
from rag_lab.retrieval.colbert_backend import ColBERTVectorStoreShim

with open(project_root / "data" / "id_to_text.json") as f:
    id_to_text = json.load(f)

# Reuses the index already built in 08_colbert.ipynb — do NOT rebuild it
# here, colbert_index_build.py already indexed the clean 576-chunk corpus.
colbert_vectorstore = ColBERTVectorStoreShim(
    index_folder=str(project_root / "data" / "colbert_index"),
    id_to_text=id_to_text,
)
print(f"colbert backend: {len(id_to_text)} chunks")

/home/mohnish/my-jupyter-env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████████████████████████████| 199/199 [00:00<00:00, 524.40it/s]


colbert backend: 576 chunks


## Choose the generation LLM

Set `USE_LOCAL_OLLAMA = True` to point generation at the local Ollama model instead of Groq — recommended for this run given the doubled call volume (see note above). Both backends use the exact same `llm` instance, so retrieval backend is the only variable that changes between the two runs of each strategy — that's the whole point of this comparison.

In [43]:
USE_LOCAL_OLLAMA = False  

if USE_LOCAL_OLLAMA:
    from langchain_ollama import ChatOllama
    OLLAMA_BASE_URL = "http://localhost:11434" 
    llm = ChatOllama(model="qwen2.5:7b", base_url=OLLAMA_BASE_URL, temperature=0)
else:
    from langchain_groq import ChatGroq
    from langchain_google_genai import ChatGoogleGenerativeAI

    #llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash")
    llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

print(f"Generation LLM: {llm}")

Generation LLM: metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.3.17'}} profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True} client=<groq.resources.chat.completions.Completions object at 0x7f12cc6f8a50> async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7f12cc6f8f50> model_name='openai/gpt-oss-120b' temperature=1e-08 model_kwargs={} groq_api_key=SecretStr('**********')


## Strategies and backends to cross

In [7]:
from rag_lab.strategies.naive import NaiveStrategy
from rag_lab.strategies.multi_query import MultiQueryStrategy
from rag_lab.strategies.rag_fusion import RagFusionStrategy
from rag_lab.strategies.hyde import HydeStrategy
from rag_lab.strategies.step_back import StepBackStrategy
from rag_lab.strategies.decomposition import DecompositionStrategy

STRATEGIES = {
    "naive": NaiveStrategy,
    "multi_query": MultiQueryStrategy,
    "rag_fusion": RagFusionStrategy,
    "hyde": HydeStrategy,
    "step_back": StepBackStrategy,
    "decomposition": DecompositionStrategy,
}

BACKENDS = {
    "dense": dense_vectorstore,
    "colbert": colbert_vectorstore,
}

print(f"{len(STRATEGIES)} strategies x {len(BACKENDS)} backends = "
      f"{len(STRATEGIES) * len(BACKENDS)} (strategy, backend) combinations")

6 strategies x 2 backends = 12 (strategy, backend) combinations


## Load the eval set and the checkpoint

In [8]:
from rag_lab.utils import load_completed_keys, append_jsonl, call_with_backoff, pacing_delay, is_daily_quota_error

EVAL_SET_PATH = project_root / "eval" / "eval_set.json"
EVAL_RESULTS_PATH = project_root / "eval" / "eval_results_backend.jsonl"

with open(EVAL_SET_PATH) as f:
    eval_data = json.load(f)
questions = eval_data["questions"]
print(f"{len(questions)} questions loaded")

completed = load_completed_keys(
    EVAL_RESULTS_PATH,
    key_fields=("strategy", "backend", "id"),
    require_fields=("answer",),
)
print(f"{len(completed)} (strategy, backend, id) rows already completed — will be skipped")

63 questions loaded
409 (strategy, backend, id) rows already completed — will be skipped


## Context-capture helper

See the assumption flagged at the top of this notebook. This never changes what `strategy.run()` returns (the answer) — it only decides how to best approximate the contexts that answer was actually grounded in, for RAGAS scoring downstream.

In [9]:
def retrieve_contexts_for_scoring(strategy, vectorstore, query, k=4):
    '''Best-effort capture of the contexts a strategy actually used.

    Returns (contexts: list[str], context_source: str) where context_source
    tells you which tier fired, so suspect rows are easy to filter later.
    '''
    if hasattr(strategy, "last_contexts") and strategy.last_contexts:
        return [d.page_content if hasattr(d, "page_content") else str(d)
                for d in strategy.last_contexts], "strategy.last_contexts"
    if hasattr(strategy, "retriever"):
        try:
            docs = strategy.retriever.invoke(query)
            return [d.page_content for d in docs], "strategy.retriever"
        except Exception:
            pass
    docs = vectorstore.as_retriever(search_kwargs={"k": k}).invoke(query)
    return [d.page_content for d in docs], "fallback_plain_retriever"

## Main loop

Keyed by `(strategy, backend, id)`, resumable, checkpointed one row at a time. Stops immediately — across all remaining strategies and backends — the moment a hard daily-quota wall is hit, rather than continuing to log guaranteed-failures for the rest of the run (the dense-only run's biggest visible waste: hundreds of `FAILED, skipping` rows in a row once Groq's TPD ceiling was hit).

In [93]:
quota_exhausted = False

for strategy_name, StrategyClass in STRATEGIES.items():
    if quota_exhausted:
        break
    for backend_name, vectorstore in BACKENDS.items():
        if quota_exhausted:
            break
        strategy = StrategyClass(vectorstore, llm=llm)
        print(f"=== {strategy_name} / {backend_name} ===")

        for q in questions:
            key = (strategy_name, backend_name, q["id"])
            if key in completed:
                continue

            record = {
                "strategy": strategy_name,
                "backend": backend_name,
                "id": q["id"],
                "question": q["question"],
                "question_type": q.get("question_type"),
                "ground_truth": q.get("ground_truth"),
                "answer": None,
                "contexts": None,
                "context_source": None,
                "error": None,
            }
            try:
                answer = call_with_backoff(lambda: strategy.run(q["question"]))
                contexts, context_source = retrieve_contexts_for_scoring(
                    strategy, vectorstore, q["question"]
                )
                record["answer"] = answer
                record["contexts"] = contexts
                record["context_source"] = context_source
                append_jsonl(EVAL_RESULTS_PATH, record)
                completed.add(key)
                print(f"  [{strategy_name}/{backend_name}] {q['id']} done")
            except Exception as e:
                record["error"] = str(e)
                append_jsonl(EVAL_RESULTS_PATH, record)
                print(f"  [{strategy_name}/{backend_name}] {q['id']} FAILED, skipping: {e}")
                if is_daily_quota_error(e):
                    print(f"Hard daily-quota wall hit on {strategy_name}/{backend_name} — "
                          f"stopping the whole run here. Resume later; completed rows are "
                          f"already checkpointed and will be skipped.")
                    quota_exhausted = True
                    break

            time.sleep(pacing_delay(strategy_name))

        if quota_exhausted:
            break
    print(f"=== Finished strategy: {strategy_name} ===\n")

print("Done." if not quota_exhausted else "Stopped early on quota exhaustion — rerun this cell later to resume.")

=== naive / dense ===
=== naive / colbert ===
=== Finished strategy: naive ===

=== multi_query / dense ===
=== multi_query / colbert ===
=== Finished strategy: multi_query ===

=== rag_fusion / dense ===
=== rag_fusion / colbert ===
=== Finished strategy: rag_fusion ===

=== hyde / dense ===
=== hyde / colbert ===
=== Finished strategy: hyde ===

=== step_back / dense ===
=== step_back / colbert ===
=== Finished strategy: step_back ===

=== decomposition / dense ===
  [decomposition/dense] b2_sk_pinn_004 done
  [decomposition/dense] b2_sk_pinn_005 done
  [decomposition/dense] b2_cross_001 done
  [decomposition/dense] b2_cross_002 done
  [decomposition/dense] b2_cross_003 FAILED, skipping: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m1yxjkkve7a82nv5dyh8rra0` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 200000, Requested 7082. Please try again in 50m59.423999999s. Need more tokens? Upgrade to 

## Sanity check: completion matrix

In [92]:
import pandas as pd

rows = []
with open(EVAL_RESULTS_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

df = pd.DataFrame(rows)
ok = df[df["answer"].notna()]
matrix = ok.pivot_table(index="strategy", columns="backend", values="id", aggfunc="count", fill_value=0)
matrix = matrix.reindex(index=list(STRATEGIES.keys()), columns=list(BACKENDS.keys()), fill_value=0)
print(f"{len(questions)} questions in eval set; completed rows per (strategy, backend):")
matrix

63 questions in eval set; completed rows per (strategy, backend):


backend,dense,colbert
strategy,,
naive,63,63
multi_query,63,63
rag_fusion,63,63
hyde,63,63
step_back,63,63
decomposition,33,0
